In [4]:
#Cell-1
import pandas as pd
from pathlib import Path

data_dir = Path(r"C:\Users\bilal\Desktop\AnalyticsVidhya_Assessment\data\archive")

questions_path = data_dir / "Questions.csv"
answers_path = data_dir / "Answers.csv"
tags_path = data_dir / "Tags.csv"

print(questions_path)
print(answers_path)
print(tags_path)

C:\Users\bilal\Desktop\AnalyticsVidhya_Assessment\data\archive\Questions.csv
C:\Users\bilal\Desktop\AnalyticsVidhya_Assessment\data\archive\Answers.csv
C:\Users\bilal\Desktop\AnalyticsVidhya_Assessment\data\archive\Tags.csv


In [5]:
#Cell-2
questions = pd.read_csv(
    questions_path,
    encoding="latin1",
    usecols=["Id", "Title", "Body", "Score"]
)

answers = pd.read_csv(
    answers_path,
    encoding="latin1",
    usecols=["ParentId", "Body", "Score"]
)

tags = pd.read_csv(
    tags_path,
    encoding="latin1"
)

print("Questions:", questions.shape)
print("Answers:", answers.shape)
print("Tags:", tags.shape)

Questions: (607282, 4)
Answers: (987122, 3)
Tags: (1885078, 2)


In [6]:
#Cell-3
print(questions.memory_usage(deep=True).sum() / 1024**2)
print(answers.memory_usage(deep=True).sum() / 1024**2)
print(tags.memory_usage(deep=True).sum() / 1024**2)

835.8980598449707
751.7747774124146
42.23478603363037


In [7]:
#Cell-4
best_answers = (
    answers
    .sort_values("Score", ascending=False)
    .drop_duplicates("ParentId")
)

print(best_answers.shape)

(539238, 3)


In [8]:
#Cell-5
import psutil

print(psutil.virtual_memory())

svmem(total=8311836672, available=437813248, percent=94.7, used=7874023424, free=437813248)


In [ ]:
#Cell-6
top_questions = (
    questions
    .sort_values("Score", ascending=False)
    .head(50000)
)

print(top_questions.shape)

(50000, 4)


In [10]:
#Cell-8
best_answers = best_answers[
    best_answers["ParentId"].isin(top_questions["Id"])
]

print(best_answers.shape)

(49824, 3)


In [11]:
#Cell-9
filtered_tags = tags[
    tags["Id"].isin(top_questions["Id"])
]

print(filtered_tags.shape)

(158141, 2)


In [12]:
#Cell-10
filtered_tags["Tag"].isna().sum()

print(filtered_tags["Tag"].dtype)

filtered_tags["Tag"].sample(10)

str


564494       timeout
312749        python
7324        pyserial
954397    python-3.x
555751        python
75437       database
692040        python
216441         queue
456263        python
181698        python
Name: Tag, dtype: str

In [13]:
#Cell-11
print(filtered_tags["Tag"].isna().sum())

64


In [14]:
#Cell-12
question_tags = (
    filtered_tags
    .groupby("Id")["Tag"]
    .apply(
        lambda x: ", ".join(
            map(str, x.dropna())
        )
    )
    .reset_index()
)

print(question_tags.shape)
question_tags.head()

(50000, 2)


,Id,Tag
0,469,"python, osx, fonts, photoshop"
1,502,"python, windows, image, pdf"
2,535,"python, continuous-integration, extreme-progra..."
3,594,"python, sql, database, oracle, cx-oracle"
4,683,"python, arrays, iteration"


In [15]:
#Cell-13
print(question_tags.shape)

(50000, 2)


In [16]:
#Cell-14
merged = top_questions.merge(
    best_answers,
    left_on="Id",
    right_on="ParentId",
    how="left"
)

print(merged.shape)

(50000, 7)


In [17]:
#Cell-15
print(merged.columns.tolist())

['Id', 'Score_x', 'Title', 'Body_x', 'ParentId', 'Score_y', 'Body_y']


In [18]:
#Cell-16
merged = merged.merge(
    question_tags,
    on="Id",
    how="left"
)

print(merged.shape)
print(merged.columns.tolist())


(50000, 8)
['Id', 'Score_x', 'Title', 'Body_x', 'ParentId', 'Score_y', 'Body_y', 'Tag']


In [19]:
#Cell-17
merged = merged.rename(
    columns={
        "Body_x": "QuestionBody",
        "Body_y": "AnswerBody",
        "Score_x": "QuestionScore",
        "Score_y": "AnswerScore",
        "Tag": "Tags"
    }
)

merged.head(2)

,Id,QuestionScore,Title,QuestionBody,ParentId,AnswerScore,AnswerBody,Tags
0,231767,5524,"What does the ""yield"" keyword do?",<p>What is the use of the <code>yield</code> k...,231767.0,8384.0,"<p>To understand what <code>yield</code> does,...","python, iterator, generator, yield, coroutine"
1,100003,3219,What is a metaclass in Python?,<p>What are metaclasses? What do you use them ...,100003.0,4510.0,<h1>Classes as objects</h1>\n\n<p>Before under...,"python, oop, metaclass, python-datamodel"


In [20]:
print("Missing QuestionBody:", merged["QuestionBody"].isna().sum())
print("Missing AnswerBody:", merged["AnswerBody"].isna().sum())
print("Missing Tags:", merged["Tags"].isna().sum())

Missing QuestionBody: 0
Missing AnswerBody: 176
Missing Tags: 0


In [21]:
#Cell-18
from bs4 import BeautifulSoup

def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(str(text), "html.parser").get_text(" ")

In [22]:
#Cell-19
merged["QuestionBody"] = merged["QuestionBody"].apply(clean_html)
merged["AnswerBody"] = merged["AnswerBody"].apply(clean_html)

In [23]:
#Cell-20
merged = merged.dropna(subset=["AnswerBody"])

print(merged.shape)

(50000, 8)


In [24]:
#Cell-21
from bs4 import BeautifulSoup
import pandas as pd

def clean_html(text):
    if pd.isna(text):
        return ""

    try:
        return BeautifulSoup(
            str(text),
            "html.parser"
        ).get_text(" ", strip=True)

    except Exception:
        return str(text)

In [25]:
#Cell-22
def test_clean(text):
    try:
        return clean_html(text)
    except Exception as e:
        print("ERROR:", e)
        print(str(text)[:1000])
        return ""

for idx, text in enumerate(merged["QuestionBody"]):
    try:
        clean_html(text)
    except Exception as e:
        print("Failed at row:", idx)
        print("Question ID:", merged.iloc[idx]["Id"])
        print(e)
        break

C:\pip_temp\ipykernel_4480\675596008.py:10: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  return BeautifulSoup(
c:\Users\bilal\Desktop\Crosstab\.venv\Lib\site-packages\bs4\__init__.py:476: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document

In [26]:
#Cell-23
merged["QuestionBody"] = merged["QuestionBody"].apply(clean_html)
merged["AnswerBody"] = merged["AnswerBody"].apply(clean_html)

C:\pip_temp\ipykernel_4480\675596008.py:10: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  return BeautifulSoup(
c:\Users\bilal\Desktop\Crosstab\.venv\Lib\site-packages\bs4\__init__.py:476: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document

In [27]:
#Cell-24
import re
import pandas as pd

def clean_html(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r"<[^>]+>", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [28]:
#Cell-25
merged["QuestionBody"] = merged["QuestionBody"].apply(clean_html)
merged["AnswerBody"] = merged["AnswerBody"].apply(clean_html)

In [29]:
#Cell-26
print(merged["QuestionBody"].iloc[0][:500])

What is the use of the yield keyword in Python? What does it do? For example, I'm trying to understand this code 1 : def _get_child_candidates(self, distance, min_dist, max_dist): if self._leftchild and distance - max_dist = self._median: yield self._rightchild And this is the caller: result, candidates = list(), [self] while candidates: node = candidates.pop() distance = node._get_dist(obj) if distance = min_dist: result.extend(node._values) candidates.extend(node._get_child_candidates(distance


In [30]:
#Cell-27
print(merged["AnswerBody"].iloc[0][:500])

To understand what yield does, you must understand what generators are. And before generators come iterables . Iterables When you create a list, you can read its items one by one. Reading its items one by one is called iteration: >>> mylist = [1, 2, 3] >>> for i in mylist: ... print(i) 1 2 3 mylist is an iterable . When you use a list comprehension, you create a list, and so an iterable: >>> mylist = [x*x for x in range(3)] >>> for i in mylist: ... print(i) 0 1 4 Everything you can use " for... 


In [31]:
#Cell-28
merged.to_csv(
    "processed_stackoverflow.csv",
    index=False
)

print("Saved!")

Saved!


In [32]:
#Cell-29
from langchain_core.documents import Document
documents = []

for _, row in merged.iterrows():
    doc_text = f"""
Title: {row['Title']}

Question:
{row['QuestionBody']}

Tags:
{row['Tags']}

Answer:
{row['AnswerBody']}
"""

    documents.append(
        Document(
            page_content=doc_text,
            metadata={
                "question_id": int(row["Id"]),
                "tags": row["Tags"]
            }
        )
    )

print("Documents:", len(documents))
print(documents[0].page_content[:500])

Documents: 50000

Title: What does the "yield" keyword do?

Question:
What is the use of the yield keyword in Python? What does it do? For example, I'm trying to understand this code 1 : def _get_child_candidates(self, distance, min_dist, max_dist): if self._leftchild and distance - max_dist = self._median: yield self._rightchild And this is the caller: result, candidates = list(), [self] while candidates: node = candidates.pop() distance = node._get_dist(obj) if distance = min_dist: result.extend(node._values) 


In [33]:
#Cell-30
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\bilal\Desktop\Crosstab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 471.56it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
#Cell-31
test_embedding = embedding_model.embed_query(
    "What does the yield keyword do in Python?"
)

print(len(test_embedding))
print(test_embedding[:10])

384
[-0.08469344675540924, 0.03850563243031502, -0.0600285641849041, 0.08518708497285843, 0.022996312007308006, -0.030799075961112976, 0.04070764780044556, 0.014873217791318893, 0.005340869538486004, 0.0011731517734006047]


In [35]:
#Cell-32
sample_docs = documents[:1000]

In [36]:
#Cell-33
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=sample_docs,
    embedding=embedding_model,
    persist_directory="./chroma_test"
)

print("Chroma test created")

Chroma test created


In [37]:
#Cell-34
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke(
    "What does the yield keyword do in Python?"
)

for i, r in enumerate(results, 1):
    print(f"\nResult {i}")
    print(r.page_content[:500])


Result 1

Title: What does the "yield" keyword do?

Question:
What is the use of the yield keyword in Python? What does it do? For example, I'm trying to understand this code 1 : def _get_child_candidates(self, distance, min_dist, max_dist): if self._leftchild and distance - max_dist = self._median: yield self._rightchild And this is the caller: result, candidates = list(), [self] while candidates: node = candidates.pop() distance = node._get_dist(obj) if distance = min_dist: result.extend(node._values) 

Result 2

Title: What does the "yield" keyword do?

Question:
What is the use of the yield keyword in Python? What does it do? For example, I'm trying to understand this code 1 : def _get_child_candidates(self, distance, min_dist, max_dist): if self._leftchild and distance - max_dist = self._median: yield self._rightchild And this is the caller: result, candidates = list(), [self] while candidates: node = candidates.pop() distance = node._get_dist(obj) if distance = min_dist: result.

In [38]:
#Cell-35
queries = [
    "How do Python generators work?",
    "What is a metaclass?",
    "How do I iterate over an Oracle result set?",
    "How can I preview a PDF in Python?"
]
documents[:10000]

[Document(metadata={'question_id': 231767, 'tags': 'python, iterator, generator, yield, coroutine'}, page_content='\nTitle: What does the "yield" keyword do?\n\nQuestion:\nWhat is the use of the yield keyword in Python? What does it do? For example, I\'m trying to understand this code 1 : def _get_child_candidates(self, distance, min_dist, max_dist): if self._leftchild and distance - max_dist = self._median: yield self._rightchild And this is the caller: result, candidates = list(), [self] while candidates: node = candidates.pop() distance = node._get_dist(obj) if distance = min_dist: result.extend(node._values) candidates.extend(node._get_child_candidates(distance, min_dist, max_dist)) return result What happens when the method _get_child_candidates is called? A list is returned? A single element is returned? Is it called again? When will subsequent calls stop? 1. The code comes from Jochen Schulz (jrschulz), who made a great Python library for metric spaces. This is the link to the c

In [39]:
#Cell-36
len(documents)

50000

In [40]:
#Cell-37
results = vectorstore.similarity_search(
    "How do Python generators work?",
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:1000])


--- Result 1 ---

Title: What can you use Python generator functions for?

Question:
I'm starting to learn Python and I've come across generator functions, those that have a yield statement in them. I want to know what types of problems that these functions are really good at solving.

Tags:
python, generator

Answer:
Generators give you lazy evaluation. You use them by iterating over them, either explicitly with 'for' or implicitly by passing it to any function or construct that iterates. You can think of generators as returning multiple items, as if they return a list, but instead of returning them all at once they return them one-by-one, and the generator function is paused until the next item is requested. Generators are good for calculating large sets of results (in particular calculations involving loops themselves) where you don't know if you are going to need all results, or where you don't want to allocate the memory for all results at the same time. Or for situations where t

In [41]:
#Cell-38
query = "What is a metaclass?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for doc in results:
    print(doc.metadata)
    print(doc.page_content[:500])
    print("=" * 50)

{'question_id': 100003, 'tags': 'python, oop, metaclass, python-datamodel'}

Title: What is a metaclass in Python?

Question:
What are metaclasses? What do you use them for?

Tags:
python, oop, metaclass, python-datamodel

Answer:
Classes as objects Before understanding metaclasses, you need to master classes in Python. And Python has a very peculiar idea of what classes are, borrowed from the Smalltalk language. In most languages, classes are just pieces of code that describe how to produce an object. That's kinda true in Python too: >>> class ObjectCreator(object): ..
{'tags': 'python, oop, metaclass, python-datamodel', 'question_id': 100003}

Title: What is a metaclass in Python?

Question:
What are metaclasses? What do you use them for?

Tags:
python, oop, metaclass, python-datamodel

Answer:
Classes as objects Before understanding metaclasses, you need to master classes in Python. And Python has a very peculiar idea of what classes are, borrowed from the Smalltalk language. In mos

In [42]:
#Cell-39
retriever = vectorstore.as_retriever()

In [43]:
#Cell-40
query = "How can I run commands from Python?"

results = retriever.invoke(query)

for i, doc in enumerate(results[:3], 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:1000])


--- Result 1 ---

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> execfile( "someFile.py", variables ) >>> print variables # globals from the someFile module


--- Result 2 ---

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> execfile( "someFile.py", variables ) >>> print vari

In [44]:
#Cell-41
query = "How can I run commands from Python?"

results = retriever.invoke(query)

for i, doc in enumerate(results[:3], 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:1000])


--- Result 1 ---

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> execfile( "someFile.py", variables ) >>> print variables # globals from the someFile module


--- Result 2 ---

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> execfile( "someFile.py", variables ) >>> print vari

In [45]:
results = vectorstore.similarity_search(
    "How can I run commands from Python?",
    k=3
)

for doc in results:
    print(doc.metadata)
    print(doc.page_content[:500])
    print("=" * 50)

{'tags': 'python', 'question_id': 1027714}

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> execfile( "someFile.py", variables ) >>> print variables # globals from the someFile modul
{'question_id': 1027714, 'tags': 'python'}

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> exe

In [46]:
results = retriever.invoke(
    "How can I run commands from Python?"
)

print(len(results))

print(results[0].metadata)
print(results[0].page_content[:1000])

4
{'tags': 'python', 'question_id': 1027714}

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> execfile( "someFile.py", variables ) >>> print variables # globals from the someFile module



In [47]:
results = retriever.invoke(
    "How can I run commands from Python?"
)

print(len(results))

print(results[0].metadata)
print(results[0].page_content[:1000])

4
{'tags': 'python', 'question_id': 1027714}

Title: How to execute a file within the python interpreter?

Question:
I'm trying to execute a file with python commands from within the interpreter. EDIT: I'm trying to use variables and settings from that file, not to invoke a separate process.

Tags:
python

Answer:
Several ways. From the shell python someFile.py From inside IDLE, hit F5 . If you're typing interactively, try this. >>> variables= {} >>> execfile( "someFile.py", variables ) >>> print variables # globals from the someFile module



In [48]:
type(vectorstore)

langchain_chroma.vectorstores.Chroma

In [49]:
vectorstore._collection.count()

2000

In [50]:
print(type(vectorstore))
print(vectorstore._collection.count())

<class 'langchain_chroma.vectorstores.Chroma'>
2000


In [51]:
vectorstore = Chroma.from_documents(
    documents[:5000],
    embedding_model
)

In [52]:
print(vectorstore._collection.count())

5000


In [53]:
retriever = vectorstore.as_retriever()

results = retriever.invoke(
    "How can I run commands from Python?"
)

print(len(results))
print(results[0].metadata)

4
{'tags': 'python', 'question_id': 1027714}


In [54]:
#Cell-37
results = vectorstore.similarity_search(
    "How do Python generators work?",
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:1000])


--- Result 1 ---

Title: Understanding Generators in Python

Question:
Reading the Python cookbook at the minute and currently looking at generators. I'm finding it hard to get my head round. As I come from a Java background, is there a Java equivalent? The book was speaking about 'Producer / Consumer', however when I hear that I think of threading. Can anyone explain what a generator is and why you would use it? Without quoting any books, obviously (unless you can find a decent, simplistic answer direct from a book). Perhaps with examples, if you're feeling generous!

Tags:
python, generator

Answer:
Note: this post assumes Python 3.x syntax. † A generator is simply a function which returns an object on which you can call next , such that for every call it returns some value, until it raises a StopIteration exception, signaling that all values have been generated. Such an object is called an iterator . Normal functions return a single value using return , just like in Java. In Python

In [55]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="stackoverflow_qa",
    embedding_function=embedding_model,
    persist_directory="./chroma_db"
)

print("Persistent DB created")

Persistent DB created


In [56]:
import os
print(os.path.exists("./chroma_db"))

True


In [2]:
import shutil
import os

folders = [
    "./chroma_db",
    "./chroma"
]

for folder in folders:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Deleted: {folder}")
    else:
        print(f"Not found: {folder}")

Not found: ./chroma_db
Not found: ./chroma


In [ ]:
import os

print(os.path.exists("./chroma_db"))
print(os.path.exists("./chroma"))

False
False


In [3]:
import shutil
import os

folders = [
    "./chroma_db",
    "./chroma_test"
]

for folder in folders:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Deleted: {folder}")
    else:
        print(f"Not found: {folder}")

Not found: ./chroma_db
Not found: ./chroma_test


In [1]:
import shutil
import os

shutil.rmtree("./chroma_db")
shutil.rmtree("./chroma_test")

print("Deleted")

Deleted


In [9]:
import os

print(os.path.exists("processed_stackoverflow.csv"))
print(os.path.exists("processed_stackoverflow.parquet"))
print(os.path.exists("documents.pkl"))

True
False
False


In [10]:
import pandas as pd

df = pd.read_csv("processed_stackoverflow.csv")

print(df.shape)
df.head(2)

(50000, 8)


,Id,QuestionScore,Title,QuestionBody,ParentId,AnswerScore,AnswerBody,Tags
0,231767,5524,"What does the ""yield"" keyword do?",What is the use of the yield keyword in Python...,231767.0,8384.0,"To understand what yield does, you must unders...","python, iterator, generator, yield, coroutine"
1,100003,3219,What is a metaclass in Python?,What are metaclasses? What do you use them for?,100003.0,4510.0,Classes as objects Before understanding metacl...,"python, oop, metaclass, python-datamodel"


In [11]:
print(df.columns.tolist())

['Id', 'QuestionScore', 'Title', 'QuestionBody', 'ParentId', 'AnswerScore', 'AnswerBody', 'Tags']


In [12]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    text = f"""
Title: {row['Title']}

Question:
{row['QuestionBody']}

Tags:
{row['Tags']}

Answer:
{row['AnswerBody']}
"""

    documents.append(
        Document(
            page_content=text,
            metadata={
                "question_id": int(row["Id"]),
                "tags": str(row["Tags"])
            }
        )
    )

print("Documents:", len(documents))

Documents: 50000


In [13]:
print(type(documents))
print(len(documents))
print(documents[0].metadata)

<class 'list'>
50000
{'question_id': 231767, 'tags': 'python, iterator, generator, yield, coroutine'}


In [15]:
"vectorstore" in globals()

False

In [16]:
import shutil
import os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Deleted")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: './chroma_db\\a7916413-1e19-42db-9a5b-f38bc6d7d5ca\\data_level0.bin'

In [17]:
print("vectorstore" in globals())
print("embedding_model" in globals())

False
False


In [2]:
%whos

Interactive namespace is empty.


In [3]:
print("df" in globals())
print("documents" in globals())
print("embedding_model" in globals())

False
False
False


In [4]:
print("df" in globals())
print("documents" in globals())
print(df.shape if "df" in globals() else "No df")

False
False
No df


In [5]:
import os

print(os.listdir("."))
print(os.listdir("./data"))

['.venv', '01_data_preparation.ipynb', 'data', 'inspect_dataset.ipynb', 'processed_stackoverflow.csv']
['archive']


In [7]:
import pandas as pd

df = pd.read_csv("processed_stackoverflow.csv")

print(df.shape)
df.head(2)

(50000, 8)


,Id,QuestionScore,Title,QuestionBody,ParentId,AnswerScore,AnswerBody,Tags
0,231767,5524,"What does the ""yield"" keyword do?",What is the use of the yield keyword in Python...,231767.0,8384.0,"To understand what yield does, you must unders...","python, iterator, generator, yield, coroutine"
1,100003,3219,What is a metaclass in Python?,What are metaclasses? What do you use them for?,100003.0,4510.0,Classes as objects Before understanding metacl...,"python, oop, metaclass, python-datamodel"


In [8]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():

    content = f"""
Title: {row['Title']}

Question:
{row['QuestionBody']}

Tags:
{row['Tags']}

Answer:
{row['AnswerBody']}
"""

    documents.append(
        Document(
            page_content=content,
            metadata={
                "question_id": int(row["Id"]),
                "tags": str(row["Tags"])
            }
        )
    )

print("Documents:", len(documents))

Documents: 50000


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\bilal\Desktop\Crosstab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 295.54it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
print(len(documents))
print(documents[0].metadata)

50000
{'question_id': 231767, 'tags': 'python, iterator, generator, yield, coroutine'}


In [11]:
from langchain_chroma import Chroma

test_store = Chroma.from_documents(
    documents[:100],
    embedding_model
)

print(test_store._collection.count())

100


In [12]:
del test_store

In [13]:
from langchain_chroma import Chroma

persist_directory = "./chroma_db"
batch_size = 5000

In [15]:
for i in range(0, len(documents), batch_size):

    batch = documents[i:i + batch_size]

    print(f"\nProcessing batch {i} to {i + len(batch)}")

    if i == 0:
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embedding_model,
            persist_directory=persist_directory
        )
    else:
        vectorstore.add_documents(batch)

    print(
        f"Current count: {vectorstore._collection.count()}"
    )

print("\nEmbedding Complete!")
print("Final Count:", vectorstore._collection.count())


Processing batch 0 to 5000
Current count: 5000

Processing batch 5000 to 10000
Current count: 10000

Processing batch 10000 to 15000
Current count: 15000

Processing batch 15000 to 20000
Current count: 20000

Processing batch 20000 to 25000
Current count: 25000

Processing batch 25000 to 30000
Current count: 30000

Processing batch 30000 to 35000
Current count: 35000

Processing batch 35000 to 40000
Current count: 40000

Processing batch 40000 to 45000
Current count: 45000

Processing batch 45000 to 50000
Current count: 50000

Embedding Complete!
Final Count: 50000


In [16]:
print(vectorstore._collection.count())

50000


In [17]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [18]:
query = "How do Python generators work?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:1000])


--- Result 1 ---

Title: Understanding Generators in Python

Question:
Reading the Python cookbook at the minute and currently looking at generators. I'm finding it hard to get my head round. As I come from a Java background, is there a Java equivalent? The book was speaking about 'Producer / Consumer', however when I hear that I think of threading. Can anyone explain what a generator is and why you would use it? Without quoting any books, obviously (unless you can find a decent, simplistic answer direct from a book). Perhaps with examples, if you're feeling generous!

Tags:
python, generator

Answer:
Note: this post assumes Python 3.x syntax. † A generator is simply a function which returns an object on which you can call next , such that for every call it returns some value, until it raises a StopIteration exception, signaling that all values have been generated. Such an object is called an iterator . Normal functions return a single value using return , just like in Java. In Python

In [19]:
import os

print(os.path.exists("./chroma_db"))

True


In [20]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model
)

In [1]:
import sys
print(sys.executable)

c:\Users\bilal\Desktop\Crosstab\.venv\Scripts\python.exe


In [2]:
import importlib
import rag_pipeline

importlib.reload(rag_pipeline)

print(dir(rag_pipeline))

c:\Users\bilal\Desktop\Crosstab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 294.71it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 342.07it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+-----------

['ChatGroq', 'Chroma', 'HuggingFaceEmbeddings', 'PromptTemplate', 'THRESHOLD', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ask_rag', 'embedding_model', 'llm', 'load_dotenv', 'prompt', 'vectorstore']


In [4]:
from rag_pipeline import ask_rag

result = ask_rag(
    "What is a Python decorator?"
)

print(result)

{'answer': 'A Python decorator is a special type of function that can modify or extend the behavior of another function. It is a higher-order function, meaning it takes another function as an argument and returns a new function that "wraps" the original function. The new function produced by the decorator is then called instead of the original function when it\'s invoked.\n\nDecorators are often used to add functionality to functions without modifying their source code. They can be used to implement aspects such as logging, authentication, caching, and error handling.\n\nHere\'s an example of a simple decorator:\n```python\ndef my_decorator(func):\n    def wrapper():\n        print("Something is happening before the function is called.")\n        func()\n        print("Something is happening after the function is called.")\n    return wrapper\n\n@my_decorator\ndef say_hello():\n    print("Hello!")\n\nsay_hello()\n```\nIn this example, the `my_decorator` function takes the `say_hello` f

In [10]:
result = ask_rag(
    "Who won FIFA World Cup 2022?"
)

print(result)

{'answer': 'I could not find the answer in the provided knowledge base.'}


In [17]:
from qdrant_client.http.models import Document

test = Document(
    text="What is a Python decorator?",
    model="sentence-transformers/all-MiniLM-L6-v2"
)

print(test)

text='What is a Python decorator?' model='sentence-transformers/all-MiniLM-L6-v2' options=None


In [18]:
import importlib.metadata

print(importlib.metadata.version("qdrant-client"))
print(importlib.metadata.version("langchain-qdrant"))

1.18.0
1.1.0
